In [1]:
# import random
# from itertools import combinations
# from collections import Counter

# # ---- Card representation ----
# RANKS = list(range(2, 15))  # 2-14, 14 = Ace
# SUITS = ['s', 'h', 'd', 'c']
# DECK = [(r, s) for r in RANKS for s in SUITS]

# # ---- Hand evaluation ----

# def best_hand_rank(cards):
#     """Return a comparable tuple representing best 5-card hand from given cards."""
#     best = None
#     for combo in combinations(cards, min(5, len(cards))):
#         rank = hand_rank(combo)
#         if best is None or rank > best:
#             best = rank
#     return best

# def hand_rank(cards):
#     ranks = sorted([r for r, s in cards], reverse=True)
#     suits = [s for r, s in cards]
#     counts = Counter(ranks)
#     rank_counts = sorted(counts.values(), reverse=True)

#     is_flush = len(set(suits)) == 1
#     is_straight = (len(set(ranks)) == 5 and ranks[0] - ranks[4] == 4)

#     # Wheel straight A-2-3-4-5
#     if set(ranks) == {14, 2, 3, 4, 5}:
#         is_straight = True
#         ranks = [5, 4, 3, 2, 1]
#         counts = Counter(ranks)

#     # Build tiebreaker: sort by frequency desc, then rank value desc
#     # This ensures paired cards always lead, kickers fall to the end
#     tiebreaker = sorted(ranks, key=lambda r: (counts[r], r), reverse=True)

#     if is_straight and is_flush:
#         return (8, tiebreaker)
#     if rank_counts[0] == 4:
#         return (7, tiebreaker)
#     if rank_counts[:2] == [3, 2]:
#         return (6, tiebreaker)
#     if is_flush:
#         return (5, tiebreaker)
#     if is_straight:
#         return (4, tiebreaker)
#     if rank_counts[0] == 3:
#         return (3, tiebreaker)
#     if rank_counts[:2] == [2, 2]:
#         return (2, tiebreaker)
#     if rank_counts[0] == 2:
#         return (1, tiebreaker)
#     return (0, tiebreaker)

# # ---- Context-aware rule generation ----

# def detect_draws(my_hand):
#     """Look at current hand and return what draws are worth chasing."""
#     if not my_hand:
#         return {'any'}

#     ranks = [r for r, s in my_hand]
#     suits = [s for r, s in my_hand]
#     suit_counts = Counter(suits)
#     rank_counts = Counter(ranks)

#     draws = set()

#     # Flush draw: 2+ cards of same suit
#     for suit, count in suit_counts.items():
#         if count >= 2:
#             draws.add(('flush', suit))

#     # Pair/trips/quads draw: 2+ cards of same rank
#     for rank, count in rank_counts.items():
#         if count >= 2:
#             draws.add(('rank', rank))

#     # Straight draw: 2+ cards within any 5-rank window
#     for low in range(2, 11):  # windows 2-6 through 10-14
#         window = set(range(low, low + 5))
#         overlap = len(window & set(ranks))
#         if overlap >= 2:
#             draws.add(('straight', low))

#     # Always consider high cards as a fallback
#     draws.add('high')

#     return draws

# def contextual_rules(my_hand, remaining_deck):
#     """Generate only rules relevant to the current hand's draws."""
#     draws = detect_draws(my_hand)
#     rules = []

#     for draw in draws:
#         if draw == 'any' or draw == 'high':
#             # Early game or no clear draw: target high cards only
#             for rank in range(10, 15):  # 10, J, Q, K, A
#                 rules.append((f'rank_gte_{rank}', lambda c, r=rank: c[0] >= r))
#             rules.append(('any', lambda c: True))

#         elif draw[0] == 'flush':
#             suit = draw[1]
#             # Any card of the flush suit, or high cards of that suit
#             rules.append((f'suit_{suit}', lambda c, s=suit: c[1] == s))
#             for rank in range(10, 15):
#                 rules.append((f'suit_{suit}_rank_gte_{rank}',
#                               lambda c, s=suit, r=rank: c[1] == s and c[0] >= r))

#         elif draw[0] == 'rank':
#             rank = draw[1]
#             # Want more copies of this rank
#             rules.append((f'rank_eq_{rank}', lambda c, r=rank: c[0] == r))

#         elif draw[0] == 'straight':
#             low = draw[1]
#             hand_ranks = set(r for r, s in my_hand)
#             needed = [r for r in range(low, low + 5) if r not in hand_ranks]
#             for rank in needed:
#                 rules.append((f'rank_eq_{rank}', lambda c, r=rank: c[0] == r))

#     # Deduplicate by name, preserving order
#     seen = set()
#     deduped = []
#     for name, fn in rules:
#         if name not in seen:
#             seen.add(name)
#             deduped.append((name, fn))

#     return deduped

# # ---- Mode toggle ----
# # FAST_MODE=True:  samples=10, n_simulations=50  → ~2-5 min,   noisy but runnable
# # FAST_MODE=False: samples=25, n_simulations=100 → ~30-80 min, good for overnight
# FAST_MODE = True

# SAMPLES = 10 if FAST_MODE else 25
# N_SIMULATIONS = 20 if FAST_MODE else 100

# # ---- Simulation ----

# def expected_value(rule_fn, my_hand, dealer_hand, remaining_deck, picks_left, samples=None):
#     samples = samples if samples is not None else SAMPLES
#     matching = [c for c in remaining_deck if rule_fn(c)]
#     if not matching:
#         return -1  # invalid rule

#     total = 0
#     for _ in range(samples):
#         shuffled = list(remaining_deck)
#         random.shuffle(shuffled)

#         # Find first matching card
#         target_idx = next(i for i, c in enumerate(shuffled) if rule_fn(c))
#         target_card = shuffled[target_idx]
#         cards_to_dealer = shuffled[:target_idx]
#         new_remaining = shuffled[target_idx + 1:]

#         new_my_hand = my_hand + [target_card]
#         new_dealer_hand = dealer_hand + cards_to_dealer

#         total += dp(new_my_hand, new_dealer_hand, new_remaining, picks_left - 1)

#     return total / samples

# def dp(my_hand, dealer_hand, remaining_deck, picks_left):
#     # Terminal: player has 5 cards, no picks left, or deck empty
#     if picks_left == 0 or len(remaining_deck) == 0 or len(my_hand) == 5:
#         return evaluate_terminal(my_hand, dealer_hand, remaining_deck)

#     best_ev = -1
#     best_rule_name = None

#     rules = contextual_rules(my_hand, remaining_deck)
#     for rule_name, rule_fn in rules:
#         matching = [c for c in remaining_deck if rule_fn(c)]
#         if not matching:
#             continue
#         ev = expected_value(rule_fn, my_hand, dealer_hand, remaining_deck, picks_left)
#         if ev > best_ev:
#             best_ev = ev
#             best_rule_name = rule_name

#     return best_ev  # return win probability

# def evaluate_terminal(my_hand, dealer_hand, remaining_deck, n_simulations=None):
#     n_simulations = n_simulations if n_simulations is not None else N_SIMULATIONS
#     if len(my_hand) < 5:
#         # Player doesn't have 5 cards — they lose
#         return 0

#     wins = 0
#     for _ in range(n_simulations):
#         deck_copy = list(remaining_deck)
#         random.shuffle(deck_copy)
#         needed = max(0, 8 - len(dealer_hand))
#         final_dealer = dealer_hand + deck_copy[:needed]
#         if best_hand_rank(my_hand) > best_hand_rank(final_dealer):
#             wins += 1

#     return wins / n_simulations

# # ---- Entry point ----

# def find_best_rule(my_hand, dealer_hand, remaining_deck, picks_left):
#     best_ev = -1
#     best_rule_name = None

#     rules = contextual_rules(my_hand, remaining_deck)
#     for rule_name, rule_fn in rules:
#         matching = [c for c in remaining_deck if rule_fn(c)]
#         if not matching:
#             continue
#         ev = expected_value(rule_fn, my_hand, dealer_hand, remaining_deck, picks_left)
#         if ev > best_ev:
#             best_ev = ev
#             best_rule_name = rule_name

#     return best_rule_name, best_ev

# # ---- Example usage ----

# if __name__ == '__main__':
#     my_hand = []
#     dealer_hand = []
#     remaining_deck = list(DECK)
#     picks_left = 5

#     rule, ev = find_best_rule(my_hand, dealer_hand, remaining_deck, picks_left)
#     print(f"Best rule: {rule}, estimated win probability: {ev:.3f}")


picks left = 2

In [2]:
# import random
# from itertools import combinations
# from collections import Counter

# # ---- Card representation ----
# RANKS = list(range(2, 15))  # 2-14, 14 = Ace
# SUITS = ['s', 'h', 'd', 'c']
# DECK = [(r, s) for r in RANKS for s in SUITS]

# # ---- Hand evaluation ----

# def best_hand_rank(cards):
#     best = None
#     for combo in combinations(cards, min(5, len(cards))):
#         rank = hand_rank(combo)
#         if best is None or rank > best:
#             best = rank
#     return best

# def hand_rank(cards):
#     ranks = sorted([r for r, s in cards], reverse=True)
#     suits = [s for r, s in cards]
#     counts = Counter(ranks)
#     rank_counts = sorted(counts.values(), reverse=True)

#     is_flush = len(set(suits)) == 1
#     is_straight = (len(set(ranks)) == 5 and ranks[0] - ranks[4] == 4)

#     if set(ranks) == {14, 2, 3, 4, 5}:
#         is_straight = True
#         ranks = [5, 4, 3, 2, 1]
#         counts = Counter(ranks)

#     tiebreaker = sorted(ranks, key=lambda r: (counts[r], r), reverse=True)

#     if is_straight and is_flush:
#         return (8, tiebreaker)
#     if rank_counts[0] == 4:
#         return (7, tiebreaker)
#     if rank_counts[:2] == [3, 2]:
#         return (6, tiebreaker)
#     if is_flush:
#         return (5, tiebreaker)
#     if is_straight:
#         return (4, tiebreaker)
#     if rank_counts[0] == 3:
#         return (3, tiebreaker)
#     if rank_counts[:2] == [2, 2]:
#         return (2, tiebreaker)
#     if rank_counts[0] == 2:
#         return (1, tiebreaker)
#     return (0, tiebreaker)

# # ---- Context-aware rule generation ----

# def detect_draws(my_hand):
#     if not my_hand:
#         return {'any'}

#     ranks = [r for r, s in my_hand]
#     suits = [s for r, s in my_hand]
#     suit_counts = Counter(suits)
#     rank_counts = Counter(ranks)
#     draws = set()

#     for suit, count in suit_counts.items():
#         if count >= 2:
#             draws.add(('flush', suit))

#     for rank, count in rank_counts.items():
#         if count >= 2:
#             draws.add(('rank', rank))

#     for low in range(2, 11):
#         window = set(range(low, low + 5))
#         if len(window & set(ranks)) >= 2:
#             draws.add(('straight', low))

#     draws.add('high')
#     return draws

# def contextual_rules(my_hand, remaining_deck):
#     draws = detect_draws(my_hand)
#     rules = []

#     for draw in draws:
#         if draw == 'any' or draw == 'high':
#             for rank in range(10, 15):
#                 rules.append((f'rank_gte_{rank}', lambda c, r=rank: c[0] >= r))
#             rules.append(('any', lambda c: True))

#         elif draw[0] == 'flush':
#             suit = draw[1]
#             rules.append((f'suit_{suit}', lambda c, s=suit: c[1] == s))
#             for rank in range(10, 15):
#                 rules.append((f'suit_{suit}_rank_gte_{rank}',
#                               lambda c, s=suit, r=rank: c[1] == s and c[0] >= r))

#         elif draw[0] == 'rank':
#             rank = draw[1]
#             rules.append((f'rank_eq_{rank}', lambda c, r=rank: c[0] == r))

#         elif draw[0] == 'straight':
#             low = draw[1]
#             hand_ranks = set(r for r, s in my_hand)
#             for rank in range(low, low + 5):
#                 if rank not in hand_ranks:
#                     rules.append((f'rank_eq_{rank}', lambda c, r=rank: c[0] == r))

#     seen = set()
#     deduped = []
#     for name, fn in rules:
#         if name not in seen:
#             seen.add(name)
#             deduped.append((name, fn))
#     return deduped

# # ---- Settings ----
# # LOOKAHEAD_DEPTH: how many picks to optimize fully before random fill
# # Keep at 2 for fast runtime (~seconds), 3 for slower but more accurate
# LOOKAHEAD_DEPTH = 2
# SAMPLES = 5
# N_SIMULATIONS = 20

# # ---- Simulation ----

# def evaluate_terminal(my_hand, dealer_hand, remaining_deck, n_simulations=None):
#     """
#     Evaluate a potentially incomplete hand by filling remaining cards randomly.
#     This allows shallow lookahead to produce meaningful EV estimates even
#     when the hand isn't complete after LOOKAHEAD_DEPTH picks.
#     """
#     n_simulations = n_simulations if n_simulations is not None else N_SIMULATIONS
#     wins = 0

#     for _ in range(n_simulations):
#         deck_copy = list(remaining_deck)
#         random.shuffle(deck_copy)

#         # Fill player hand to 5 with random cards if incomplete
#         final_my_hand = list(my_hand)
#         while len(final_my_hand) < 5 and deck_copy:
#             final_my_hand.append(deck_copy.pop(0))

#         if len(final_my_hand) < 5:
#             continue  # deck exhausted, skip

#         # Fill dealer to 8
#         needed = max(0, 8 - len(dealer_hand))
#         final_dealer = dealer_hand + deck_copy[:needed]

#         if best_hand_rank(final_my_hand) > best_hand_rank(final_dealer):
#             wins += 1

#     return wins / n_simulations

# def expected_value(rule_fn, my_hand, dealer_hand, remaining_deck, picks_left, samples=None):
#     samples = samples if samples is not None else SAMPLES
#     matching = [c for c in remaining_deck if rule_fn(c)]
#     if not matching:
#         return -1

#     total = 0
#     for _ in range(samples):
#         shuffled = list(remaining_deck)
#         random.shuffle(shuffled)

#         target_idx = next(i for i, c in enumerate(shuffled) if rule_fn(c))
#         target_card = shuffled[target_idx]
#         cards_to_dealer = shuffled[:target_idx]
#         new_remaining = shuffled[target_idx + 1:]

#         new_my_hand = my_hand + [target_card]
#         new_dealer_hand = dealer_hand + cards_to_dealer

#         total += dp(new_my_hand, new_dealer_hand, new_remaining, picks_left - 1)

#     return total / samples

# def dp(my_hand, dealer_hand, remaining_deck, picks_left):
#     # Terminal: lookahead exhausted, hand full, or deck empty
#     # Incomplete hands are filled randomly inside evaluate_terminal
#     if picks_left == 0 or len(remaining_deck) == 0 or len(my_hand) == 5:
#         return evaluate_terminal(my_hand, dealer_hand, remaining_deck)

#     best_ev = -1
#     rules = contextual_rules(my_hand, remaining_deck)
#     for rule_name, rule_fn in rules:
#         if not any(rule_fn(c) for c in remaining_deck):
#             continue
#         ev = expected_value(rule_fn, my_hand, dealer_hand, remaining_deck, picks_left)
#         if ev > best_ev:
#             best_ev = ev

#     return best_ev

# # ---- Entry point ----

# def find_best_rule(my_hand, dealer_hand, remaining_deck, picks_left=None):
#     picks_left = picks_left if picks_left is not None else LOOKAHEAD_DEPTH
#     best_ev = -1
#     best_rule_name = None

#     rules = contextual_rules(my_hand, remaining_deck)
#     print(f"Evaluating {len(rules)} rules with lookahead depth {picks_left}...")

#     for rule_name, rule_fn in rules:
#         if not any(rule_fn(c) for c in remaining_deck):
#             continue
#         ev = expected_value(rule_fn, my_hand, dealer_hand, remaining_deck, picks_left)
#         print(f"  {rule_name}: EV = {ev:.3f}")
#         if ev > best_ev:
#             best_ev = ev
#             best_rule_name = rule_name

#     return best_rule_name, best_ev

# # ---- Example usage ----

# if __name__ == '__main__':
#     print("=" * 50)
#     print("Finding best opening rule (empty hand)...")
#     print("=" * 50)
#     my_hand = []
#     dealer_hand = []
#     remaining_deck = list(DECK)

#     rule, ev = find_best_rule(my_hand, dealer_hand, remaining_deck)
#     print(f"\nBest opening rule: {rule}")
#     print(f"Estimated win probability: {ev:.3f}")

#     # print("\n" + "=" * 50)
#     # print("Mid-game example: hand = [Ks, Qs]")
#     # print("=" * 50)
#     # my_hand = [(13, 's'), (12, 's')]
#     # remaining_deck = [c for c in DECK if c not in my_hand]

#     # rule, ev = find_best_rule(my_hand, dealer_hand, remaining_deck)
#     # print(f"\nBest rule: {rule}")
#     # print(f"Estimated win probability: {ev:.3f}")

first rule w suit s added

In [3]:
# import random
# from itertools import combinations
# from collections import Counter

# # ---- Card representation ----
# RANKS = list(range(2, 15))  # 2-14, 14 = Ace
# SUITS = ['s', 'h', 'd', 'c']
# DECK = [(r, s) for r in RANKS for s in SUITS]

# # ---- Hand evaluation ----

# def best_hand_rank(cards):
#     best = None
#     for combo in combinations(cards, min(5, len(cards))):
#         rank = hand_rank(combo)
#         if best is None or rank > best:
#             best = rank
#     return best

# def hand_rank(cards):
#     ranks = sorted([r for r, s in cards], reverse=True)
#     suits = [s for r, s in cards]
#     counts = Counter(ranks)
#     rank_counts = sorted(counts.values(), reverse=True)

#     is_flush = len(set(suits)) == 1
#     is_straight = (len(set(ranks)) == 5 and ranks[0] - ranks[4] == 4)

#     if set(ranks) == {14, 2, 3, 4, 5}:
#         is_straight = True
#         ranks = [5, 4, 3, 2, 1]
#         counts = Counter(ranks)

#     tiebreaker = sorted(ranks, key=lambda r: (counts[r], r), reverse=True)

#     if is_straight and is_flush:
#         return (8, tiebreaker)
#     if rank_counts[0] == 4:
#         return (7, tiebreaker)
#     if rank_counts[:2] == [3, 2]:
#         return (6, tiebreaker)
#     if is_flush:
#         return (5, tiebreaker)
#     if is_straight:
#         return (4, tiebreaker)
#     if rank_counts[0] == 3:
#         return (3, tiebreaker)
#     if rank_counts[:2] == [2, 2]:
#         return (2, tiebreaker)
#     if rank_counts[0] == 2:
#         return (1, tiebreaker)
#     return (0, tiebreaker)

# # ---- Context-aware rule generation ----

# def detect_draws(my_hand):
#     if not my_hand:
#         return {'any'}

#     ranks = [r for r, s in my_hand]
#     suits = [s for r, s in my_hand]
#     suit_counts = Counter(suits)
#     rank_counts = Counter(ranks)
#     draws = set()

#     for suit, count in suit_counts.items():
#         if count >= 2:
#             draws.add(('flush', suit))

#     for rank, count in rank_counts.items():
#         if count >= 2:
#             draws.add(('rank', rank))

#     for low in range(2, 11):
#         window = set(range(low, low + 5))
#         if len(window & set(ranks)) >= 2:
#             draws.add(('straight', low))

#     draws.add('high')
#     return draws

# def contextual_rules(my_hand, remaining_deck):
#     draws = detect_draws(my_hand)
#     rules = []

#     for draw in draws:
#         if draw == 'any' or draw == 'high':
#             for rank in range(10, 15):
#                 rules.append((f'rank_gte_{rank}', lambda c, r=rank: c[0] >= r))
#             rules.append(('any', lambda c: True))
#             # Explicitly add suit_s as opening option — flush strategy
#             # not detected until 2+ cards of same suit exist
#             rules.append(('suit_s', lambda c: c[1] == 's'))

#         elif draw[0] == 'flush':
#             suit = draw[1]
#             rules.append((f'suit_{suit}', lambda c, s=suit: c[1] == s))
#             for rank in range(10, 15):
#                 rules.append((f'suit_{suit}_rank_gte_{rank}',
#                               lambda c, s=suit, r=rank: c[1] == s and c[0] >= r))

#         elif draw[0] == 'rank':
#             rank = draw[1]
#             rules.append((f'rank_eq_{rank}', lambda c, r=rank: c[0] == r))

#         elif draw[0] == 'straight':
#             low = draw[1]
#             hand_ranks = set(r for r, s in my_hand)
#             for rank in range(low, low + 5):
#                 if rank not in hand_ranks:
#                     rules.append((f'rank_eq_{rank}', lambda c, r=rank: c[0] == r))

#     seen = set()
#     deduped = []
#     for name, fn in rules:
#         if name not in seen:
#             seen.add(name)
#             deduped.append((name, fn))
#     return deduped

# # ---- Settings ----
# # LOOKAHEAD_DEPTH: how many picks to optimize fully before random fill
# # Keep at 2 for fast runtime (~seconds), 3 for slower but more accurate
# LOOKAHEAD_DEPTH = 2
# SAMPLES = 5
# N_SIMULATIONS = 20

# # ---- Simulation ----

# def evaluate_terminal(my_hand, dealer_hand, remaining_deck, n_simulations=None):
#     """
#     Evaluate a potentially incomplete hand by filling remaining cards randomly.
#     This allows shallow lookahead to produce meaningful EV estimates even
#     when the hand isn't complete after LOOKAHEAD_DEPTH picks.
#     """
#     n_simulations = n_simulations if n_simulations is not None else N_SIMULATIONS
#     wins = 0

#     for _ in range(n_simulations):
#         deck_copy = list(remaining_deck)
#         random.shuffle(deck_copy)

#         # Fill player hand to 5 with random cards if incomplete
#         final_my_hand = list(my_hand)
#         while len(final_my_hand) < 5 and deck_copy:
#             final_my_hand.append(deck_copy.pop(0))

#         if len(final_my_hand) < 5:
#             continue  # deck exhausted, skip

#         # Fill dealer to 8
#         needed = max(0, 8 - len(dealer_hand))
#         final_dealer = dealer_hand + deck_copy[:needed]

#         if best_hand_rank(final_my_hand) > best_hand_rank(final_dealer):
#             wins += 1

#     return wins / n_simulations

# def expected_value(rule_fn, my_hand, dealer_hand, remaining_deck, picks_left, samples=None):
#     samples = samples if samples is not None else SAMPLES
#     matching = [c for c in remaining_deck if rule_fn(c)]
#     if not matching:
#         return -1

#     total = 0
#     for _ in range(samples):
#         shuffled = list(remaining_deck)
#         random.shuffle(shuffled)

#         target_idx = next(i for i, c in enumerate(shuffled) if rule_fn(c))
#         target_card = shuffled[target_idx]
#         cards_to_dealer = shuffled[:target_idx]
#         new_remaining = shuffled[target_idx + 1:]

#         new_my_hand = my_hand + [target_card]
#         new_dealer_hand = dealer_hand + cards_to_dealer

#         total += dp(new_my_hand, new_dealer_hand, new_remaining, picks_left - 1)

#     return total / samples

# def dp(my_hand, dealer_hand, remaining_deck, picks_left):
#     # Terminal: lookahead exhausted, hand full, or deck empty
#     # Incomplete hands are filled randomly inside evaluate_terminal
#     if picks_left == 0 or len(remaining_deck) == 0 or len(my_hand) == 5:
#         return evaluate_terminal(my_hand, dealer_hand, remaining_deck)

#     best_ev = -1
#     rules = contextual_rules(my_hand, remaining_deck)
#     for rule_name, rule_fn in rules:
#         if not any(rule_fn(c) for c in remaining_deck):
#             continue
#         ev = expected_value(rule_fn, my_hand, dealer_hand, remaining_deck, picks_left)
#         if ev > best_ev:
#             best_ev = ev

#     return best_ev

# # ---- Entry point ----

# def find_best_rule(my_hand, dealer_hand, remaining_deck, picks_left=None):
#     picks_left = picks_left if picks_left is not None else LOOKAHEAD_DEPTH
#     best_ev = -1
#     best_rule_name = None

#     rules = contextual_rules(my_hand, remaining_deck)
#     print(f"Evaluating {len(rules)} rules with lookahead depth {picks_left}...")

#     for rule_name, rule_fn in rules:
#         if not any(rule_fn(c) for c in remaining_deck):
#             continue
#         ev = expected_value(rule_fn, my_hand, dealer_hand, remaining_deck, picks_left)
#         print(f"  {rule_name}: EV = {ev:.3f}")
#         if ev > best_ev:
#             best_ev = ev
#             best_rule_name = rule_name

#     return best_rule_name, best_ev

# # ---- Example usage ----

# if __name__ == '__main__':
#     print("=" * 50)
#     print("Finding best opening rule (empty hand)...")
#     print("=" * 50)
#     my_hand = []
#     dealer_hand = []
#     remaining_deck = list(DECK)

#     rule, ev = find_best_rule(my_hand, dealer_hand, remaining_deck)
#     print(f"\nBest opening rule: {rule}")
#     print(f"Estimated win probability: {ev:.3f}")

#     print("\n" + "=" * 50)
#     print("Mid-game example: hand = [Ks, Qs]")
#     print("=" * 50)
#     my_hand = [(13, 's'), (12, 's')]
#     remaining_deck = [c for c in DECK if c not in my_hand]

#     rule, ev = find_best_rule(my_hand, dealer_hand, remaining_deck)
#     print(f"\nBest rule: {rule}")
#     print(f"Estimated win probability: {ev:.3f}")

fixed complexity, shuffle at top only

In [4]:
import random
from itertools import combinations
from collections import Counter

# ---- Card representation ----
RANKS = list(range(2, 15))  # 2-14, 14 = Ace
SUITS = ['s', 'h', 'd', 'c']
DECK = [(r, s) for r in RANKS for s in SUITS]

RANK_NAMES = {2:'2', 3:'3', 4:'4', 5:'5', 6:'6', 7:'7', 8:'8',
              9:'9', 10:'10', 11:'J', 12:'Q', 13:'K', 14:'A'}

def card_str(card):
    r, s = card
    return f"{RANK_NAMES[r]}{s}"

def hand_str(hand):
    return '[' + ', '.join(card_str(c) for c in hand) + ']'

# ---- Hand evaluation ----
# All O(n) — no combinations needed.

def best_hand_rank(cards):
    if not cards:
        return (0, [])

    ranks = sorted([r for r, s in cards], reverse=True)
    rank_counts = Counter(r for r, s in cards)
    suit_counts = Counter(s for r, s in cards)
    rank_set = set(ranks)

    # --- Flush suit (need >= 5 of same suit) ---
    flush_suit = next((s for s, c in suit_counts.items() if c >= 5), None)
    flush_ranks = sorted((r for r, s in cards if s == flush_suit), reverse=True) if flush_suit else []

    # --- Straight finder: returns highest top-card or None ---
    def best_straight(rank_set):
        # Ace can play low (as 1) for wheel: A-2-3-4-5 -> top = 5
        rs = rank_set | ({1} if 14 in rank_set else set())
        for top in range(14, 4, -1):
            window = set(range(top - 4, top + 1))
            if window <= rs:
                return top  # wheel returns 5
        return None

    straight_top = best_straight(rank_set)
    sf_top = best_straight(set(flush_ranks)) if flush_suit else None

    # --- Count-based helpers ---
    counts_desc = sorted(rank_counts.items(), key=lambda x: (x[1], x[0]), reverse=True)
    # groups by count: e.g. [(14,3),(9,2),(7,1)]
    by_count = lambda n: [r for r, c in counts_desc if c == n]

    quads   = by_count(4)
    trips   = by_count(3)
    pairs   = by_count(2)
    singles = by_count(1)

    # Tiebreaker helpers
    def kickers(exclude_ranks, n):
        return sorted([r for r in ranks if r not in exclude_ranks], reverse=True)[:n]

    # --- Evaluate from best to worst ---

    # 8: Straight flush
    if sf_top:
        return (8, [sf_top])

    # 7: Four of a kind
    if quads:
        q = quads[0]
        return (7, [q] + kickers({q}, 1))

    # 6: Full house — best trips + best pair (trips can serve as pair if two trips)
    if trips:
        t = trips[0]
        # second trip or best pair acts as the pair
        pair_rank = trips[1] if len(trips) > 1 else (pairs[0] if pairs else None)
        if pair_rank is not None:
            return (6, [t, pair_rank])

    # 5: Flush — top 5 cards of flush suit
    if flush_suit:
        return (5, flush_ranks[:5])

    # 4: Straight
    if straight_top:
        return (4, [straight_top])

    # 3: Three of a kind
    if trips:
        t = trips[0]
        return (3, [t] + kickers({t}, 2))

    # 2: Two pair — top two pairs + kicker
    if len(pairs) >= 2:
        p1, p2 = pairs[0], pairs[1]
        return (2, [p1, p2] + kickers({p1, p2}, 1))

    # 1: One pair
    if pairs:
        p = pairs[0]
        return (1, [p] + kickers({p}, 3))

    # 0: High card
    return (0, ranks[:5])

# ---- Draw detection & rule generation ----

def detect_draws(my_hand):
    if not my_hand:
        return {'any'}

    ranks = [r for r, s in my_hand]
    suits = [s for r, s in my_hand]
    suit_counts = Counter(suits)
    rank_counts = Counter(ranks)
    draws = set()

    for suit, count in suit_counts.items():
        if count >= 2:
            draws.add(('flush', suit))

    for rank, count in rank_counts.items():
        if count >= 2:
            draws.add(('rank', rank))

    for low in range(2, 11):
        window = set(range(low, low + 5))
        if len(window & set(ranks)) >= 2:
            draws.add(('straight', low))

    draws.add('high')
    return draws

def contextual_rules(my_hand, remaining_deck):
    draws = detect_draws(my_hand)
    rules = []

    for draw in draws:
        if draw == 'any' or draw == 'high':
            for rank in range(10, 15):
                rules.append((f'rank_gte_{rank}', lambda c, r=rank: c[0] >= r))
            rules.append(('any', lambda c: True))

        elif draw[0] == 'flush':
            suit = draw[1]
            rules.append((f'suit_{suit}', lambda c, s=suit: c[1] == s))
            for rank in range(10, 15):
                rules.append((f'suit_{suit}_rank_gte_{rank}',
                              lambda c, s=suit, r=rank: c[1] == s and c[0] >= r))

        elif draw[0] == 'rank':
            rank = draw[1]
            rules.append((f'rank_eq_{rank}', lambda c, r=rank: c[0] == r))

        elif draw[0] == 'straight':
            low = draw[1]
            hand_ranks = set(r for r, s in my_hand)
            for rank in range(low, low + 5):
                if rank not in hand_ranks:
                    rules.append((f'rank_eq_{rank}', lambda c, r=rank: c[0] == r))

    seen = set()
    deduped = []
    for name, fn in rules:
        if name not in seen:
            seen.add(name)
            deduped.append((name, fn))
    return deduped

# ---- Apply a rule to a fixed deck ----
# Cards before the matched card go to the dealer (fixed by deck order).
# Returns (matched_card, new_dealer_hand, new_remaining_deck)
# or None if no card in deck matches the rule.

def apply_rule(rule_fn, deck, dealer_hand):
    for i, card in enumerate(deck):
        if rule_fn(card):
            new_dealer = dealer_hand + deck[:i]
            new_deck = deck[i+1:]
            return card, new_dealer, new_deck
    return None

# ---- Deterministic tree search over rule choices (deck is fixed) ----

def dp(my_hand, dealer_hand, deck, picks_left):
    """
    Deterministic tree search over rule choices given a fixed deck ordering.
    Recalls contextual_rules at each node with the updated hand.
    """
    if picks_left == 0 or not deck or len(my_hand) >= 5:
        # Top up dealer to 8 cards from remaining deck
        needed = max(0, 8 - len(dealer_hand))
        final_dealer = dealer_hand + deck[:needed]
        my_rank = best_hand_rank(my_hand)
        dealer_rank = best_hand_rank(final_dealer)
        return 1.0 if my_rank > dealer_rank else 0.0

    best_ev = -1.0
    seen_cards = set()  # dedupe: different rules may match same card
    for rule_name, rule_fn in contextual_rules(my_hand, deck):
        result = apply_rule(rule_fn, deck, dealer_hand)
        if result is None:
            continue
        card, new_dealer, new_deck = result
        if card in seen_cards:
            continue
        seen_cards.add(card)
        ev = dp(my_hand + [card], new_dealer, new_deck, picks_left - 1)
        if ev > best_ev:
            best_ev = ev
    return best_ev

# ---- Top-level: shuffle once per game, run N_SIMULATIONS games ----

def find_best_rule(my_hand, dealer_hand, picks_left, n_simulations=200):
    """
    Shuffles once per simulation. Each simulation is a fully deterministic
    tree search — no resampling at any point during play.
    """
    base_deck = [c for c in DECK if c not in my_hand and c not in dealer_hand]
    rule_evs = Counter()
    rule_counts = Counter()

    for _ in range(n_simulations):
        deck = list(base_deck)
        random.shuffle(deck)  # only shuffle here

        # One-level lookahead to find which rule is best for this deck ordering
        rules = contextual_rules(my_hand, deck)
        for rule_name, rule_fn in rules:
            result = apply_rule(rule_fn, deck, dealer_hand)
            if result is None:
                continue
            card, new_dealer, new_deck = result
            ev = dp(my_hand + [card], new_dealer, new_deck, picks_left - 1)
            rule_evs[rule_name] += ev
            rule_counts[rule_name] += 1

    all_results = [
        (name, rule_evs[name] / rule_counts[name])
        for name in rule_evs
    ]
    all_results.sort(key=lambda x: x[1], reverse=True)
    best_rule, best_ev = all_results[0] if all_results else (None, -1)
    return best_rule, best_ev, all_results

if __name__ == '__main__':
    N_SIMULATIONS = 5000

    # Start of game: player has no cards, picks entire hand from scratch
    picks_left = 5
    my_hand = []
    dealer_hand = []

    print("=" * 60)
    print(f"Picks left: {picks_left}  (player starts with empty hand)")
    print()

    best_rule, best_ev, all_results = find_best_rule(
        my_hand, dealer_hand, picks_left, n_simulations=N_SIMULATIONS
    )

    print(f"  {'Rule':<35} {'EV':>6}")
    print(f"  {'-'*43}")
    for rule_name, ev in all_results:
        marker = " <-- BEST" if rule_name == best_rule else ""
        print(f"  {rule_name:<35} {ev:>6.3f}{marker}")

    print(f"\n  Best rule: {best_rule}  (EV = {best_ev:.3f})")
    print()

Picks left: 5  (player starts with empty hand)

  Rule                                    EV
  -------------------------------------------
  any                                  0.933 <-- BEST
  rank_gte_10                          0.923
  rank_gte_11                          0.909
  rank_gte_12                          0.882
  rank_gte_13                          0.829
  rank_gte_14                          0.673

  Best rule: any  (EV = 0.933)

